# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All dataset elements are referenced via their Croissant `@id` identifiers for reproducibility and clarity.

### Dataset Source
The dataset is described with a Croissant schema available at the following URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata object
metadata = dataset.metadata

# Print main dataset title and description
print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)


## 2. Data Overview
Review record sets and their associated fields in the dataset schema. All references use their `@id` as per FAIR² conventions.

In [ ]:
# List all RecordSets in the dataset
print("\nAvailable Record Sets:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    record_set_ids.append(rs.id)

# For each RecordSet, show associated Fields and their IDs
for rs in metadata.record_sets:
    print(f"\nRecordSet '@id': {rs.id}  ({getattr(rs, 'name', '-')}) fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} -- name: {field.name} (dataType: {field.data_type})")

## 3. Data Extraction
Load records from selected RecordSet(s) using their `@id`. Each row is loaded as a dictionary with keys being field `@id`s.

In [ ]:
# Example: Load all available record sets into DataFrames, indexed by their Croissant @id
dataframes = {}

for rs in metadata.record_sets:
    print(f"Loading data for RecordSet: {rs.id}")
    records = list(dataset.records(record_set=rs.id))
    if records:  # Only create a DataFrame if records exist
        dataframes[rs.id] = pd.DataFrame(records)

if dataframes:
    # Use the first RecordSet as main example
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns (@id of Fields) in main RecordSet '{main_record_set_id}':\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No records found in RecordSets.")

## 4. Exploratory Data Analysis (EDA)
Process a numeric field for simple filtering and normalization. All field references continue to use their Croissant `@id` as the column names.

**Example:** We select the first numeric field from the main record set for demonstration.

In [ ]:
import numpy as np

# Detect numeric fields by their dataType in the main RecordSet
main_rs = None
for rs in metadata.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break

numeric_fields = [field for field in main_rs.fields if field.data_type in ['Integer', 'Float', 'Number']]
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
else:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes(include=[np.number]).columns[0]

# Filter: values greater than a threshold
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field_id in df.columns:
    numeric_data = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_data > threshold].copy()
    print(f"Filtered records where '{numeric_field_id}' > {threshold}:\n", filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (numeric_data - numeric_data.mean()) / numeric_data.std(ddof=0)
    print(f"Normalized '{numeric_field_id}':\n", filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by another field (categorical)
    group_field = None
    non_numeric_fields = [field for field in main_rs.fields if field.data_type == 'Text']
    for f in non_numeric_fields:
        if f.id in df.columns:
            group_field = f.id
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of '{numeric_field_id}' by '{group_field}':\n", grouped_df.head())
else:
    print(f"No numeric field detected in RecordSet '{main_record_set_id}'.")

## 5. Visualization
Visualize the distribution of the numeric field and (optionally) its relationship to a categorical group field. Visualization uses matplotlib and seaborn for statistical insight.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        # Only plot the top 10 groups for visibility
        top_groups = filtered_df[group_field].value_counts().index[:10]
        subset = filtered_df[filtered_df[group_field].isin(top_groups)]
        sns.boxplot(x=group_field, y=numeric_field_id, data=subset)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field}' (Top 10)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to:
- Access FAIR²-compliant datasets using their Croissant schema and `mlcroissant`.
- Explore available RecordSets, Fields, and Columns using `@id` references.
- Load record data into pandas DataFrames for processing.
- Filter, normalize, and group numeric data fields.
- Visualize distributions and groupings for exploratory data analysis.

For your own studies, refer to the schema's `@id` documentation and dataset documentation for precise meaning of each field.